<a href="https://colab.research.google.com/github/Madhuanabala/SET/blob/classification/%202%20classification_after_clustering_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE

file_path = '/content/compds after clustering with GI Abs.xlsx'
data = pd.read_excel(file_path)
print(data.columns)
print(data.head())




Index(['Compound name', 'Anti-Inflammatory', 'Anti-Oxidant', 'Anti-Cancer',
       'Other Properties', 'GI absorption'],
      dtype='object')
                                       Compound name  Anti-Inflammatory  \
0                                    Betulinic acid                   1   
1                                           Galangin                  1   
2  3-Hydroxy-4-prenyl-5-methoxystilbene-2-carboxy...                  1   
3                                      δ-tocotrienol                  1   
4                                   (−)-larreatricin                  1   

   Anti-Oxidant  Anti-Cancer  Other Properties  GI absorption  
0             1            1                 1              0  
1             1            1                 1              1  
2             1            1                 0              1  
3             1            1                 1              1  
4             1            1                 1              1  


In [5]:
y = data['GI absorption']
X = data[['Anti-Oxidant', 'Anti-Cancer', 'Other Properties']]
print(X.columns)

Index(['Anti-Oxidant', 'Anti-Cancer', 'Other Properties'], dtype='object')


In [6]:
X = data.drop(columns=['Compound name', 'Anti-Inflammatory', 'GI absorption'])
y = data['GI absorption']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_smote, y_train_smote)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))
y_train_pred = rf_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy}")



Accuracy: 0.42857142857142855
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.60      0.60      0.60        15

    accuracy                           0.43        21
   macro avg       0.30      0.30      0.30        21
weighted avg       0.43      0.43      0.43        21

Training Accuracy: 0.723404255319149


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import class_weight

X = data.drop(columns=['Compound name', 'Anti-Inflammatory', 'GI absorption'])  # Drop unnecessary columns
y = data['GI absorption']
compound_names = data['Compound name']

X_train, X_test, y_train, y_test, compound_train, compound_test = train_test_split(
    X, y, compound_names, test_size=0.3, random_state=42
)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))


rf_model = RandomForestClassifier(random_state=42, class_weight=class_weights_dict)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)
best_rf_model = grid_search.best_estimator_
cv_scores = cross_val_score(best_rf_model, X_train, y_train, cv=5)
print(f"Cross-Validated Training Accuracy: {cv_scores.mean()}")
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))

y_train_pred = best_rf_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy}")


results = pd.DataFrame({
    'Compound Name': compound_test,
    'Actual GI Absorption': y_test,
    'Predicted GI Absorption': y_pred
})


results['Training Accuracy'] = train_accuracy
results['Test Accuracy'] = accuracy


excel_filename = 'gi_absorption_predictions_with_compounds.xlsx'
results.to_excel(excel_filename, index=False)

print(f"Results have been saved to {excel_filename}")


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Cross-Validated Training Accuracy: 0.7222222222222221
Test Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.71      1.00      0.83        15

    accuracy                           0.71        21
   macro avg       0.36      0.50      0.42        21
weighted avg       0.51      0.71      0.60        21

Training Accuracy: 0.8297872340425532
Results have been saved to gi_absorption_predictions_with_compounds.xlsx


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:

print(f"Number of training samples: {X_train.shape[0]}")
print(f"Number of testing samples: {X_test.shape[0]}")


Number of training samples: 47
Number of testing samples: 21


In [9]:

print(f"Number of training samples after SMOTE: {X_train_smote.shape[0]}")
print(f"Number of testing samples (unchanged): {X_test.shape[0]}")


Number of training samples after SMOTE: 78
Number of testing samples (unchanged): 21
